In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [2]:


filename='clientes credicash.csv'
df_base_inventario=cargar_archivo_csv(spark,filename,';',True)


In [ ]:
df_base_inventario.show()

+--------------+-------+--------------------+--------+-----+----+-----+---------+--------+------------+-----------+------+--------------------+--------+---------+--------------------+----------+
|          HORA| ESTADO|             CLIENTE|     DNI|MONTO|TASA|PLAZO| TELEFONO|PROMOTOR|FECHA VISITA|HORA VISITA|TIENDA|             AGENCIA|ID_VENTA| PRODUCTO|           DIRECCION|    PERFIL|
+--------------+-------+--------------------+--------+-----+----+-----+---------+--------+------------+-----------+------+--------------------+--------+---------+--------------------+----------+
|03:11:14 p. m.|Enviado|PORFIRIO VICENTE ...|44052336| 7000|49.2|   28|923225199|  CCA060|   2/05/2026|   19:00:00| CARSA|CARSA_CANETE IMPE...|   40008|CREDICASH|  CALLE HUANCAYO S/N|Diamante 1|
|09:33:41 a. m.|Enviado|MIGUEL ANGEL RUIZ...|19259438| 9500|26.8|   36|949847624|  PEC151|   2/05/2026|   16:00:00| CARSA|CARSA MOTOS_PACAS...|   39454|CREDICASH|CALLE LADISLAO ES...|Diamante 0|
|02:01:01 p. m.|Enviado|I

In [3]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
*
FROM alfcc_clientes
WHERE cl_base = 'abril 2026'
and cl_estado=1
and NUMERO_DOCUMENTO in ('44052336','19259438','70289540')
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["NUMERO_DOCUMENTO"] = (
    df_dni["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)


In [ ]:


df_dni.to_excel(ruta_archivo, index=False)

,cl_id,cl_telf1,cl_telf2,cl_telf3,cl_telf4,cl_telf5,cl_telf6,cl_telf7,cl_telf8,cl_telf9,...,TIPO_ZONA,PRODUCTO_EXTERNO,FLG_CRUCE_RECURRENTE,FLG_CRUCE,DEMANDA,PRIORIDAD,DETALLE_VARIACION,INTENSIDAD_MAX,FRESCURA,PRODUCTO_INTERNO
0,3398995,949847624,983541003,0,None,None,0,0,0,0,...,None,CREDICASH R / ELECTRO MSI,1,None,None,None,None,5,3,None
1,3418267,923225199,948053150,0,None,None,0,0,0,0,...,None,CREDICASH R / ELECTRO,1,None,None,None,None,6,4,None
2,3366940,956182452,0,0,None,None,0,0,0,0,...,None,CREDICASH R / ELECTRO,1,None,None,None,None,5,2,None


In [13]:
df_dni = df_dni.drop(columns=['cl_id'])

KeyError: "['cl_id'] not found in axis"

In [16]:

df_dni.to_sql(
    name="alfcc_clientes",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

3

In [14]:
df_dni.head()

,cl_telf1,cl_telf2,cl_telf3,cl_telf4,cl_telf5,cl_telf6,cl_telf7,cl_telf8,cl_telf9,cl_telf10,...,TIPO_ZONA,PRODUCTO_EXTERNO,FLG_CRUCE_RECURRENTE,FLG_CRUCE,DEMANDA,PRIORIDAD,DETALLE_VARIACION,INTENSIDAD_MAX,FRESCURA,PRODUCTO_INTERNO
0,949847624,983541003,0,None,None,0,0,0,0,0,...,None,CREDICASH R / ELECTRO MSI,1,None,None,None,None,5,3,None
1,923225199,948053150,0,None,None,0,0,0,0,0,...,None,CREDICASH R / ELECTRO,1,None,None,None,None,6,4,None
2,956182452,0,0,None,None,0,0,0,0,0,...,None,CREDICASH R / ELECTRO,1,None,None,None,None,5,2,None


In [12]:
df_dni['cl_base']='Mayo 2026'
df_dni['cl_mes']='Mayo 2026'
df_dni['cl_carga']='2026-05-14'
df_dni['LOTE']='2026-05-14'
df_dni['CAMPANA']='202605'
df_dni['cl_estado']=1
df_dni['estado']='ACTIVO'

df_dni[['cl_base', 'cl_mes', 'cl_carga','LOTE','CAMPANA','cl_estado','estado']].head()

,cl_base,cl_mes,cl_carga,LOTE,CAMPANA,cl_estado,estado
0,Mayo 2026,Mayo 2026,2026-05-14,2026-05-14,202605,1,ACTIVO
1,Mayo 2026,Mayo 2026,2026-05-14,2026-05-14,202605,1,ACTIVO
2,Mayo 2026,Mayo 2026,2026-05-14,2026-05-14,202605,1,ACTIVO


In [ ]:
df_base = df_base.withColumn("Campana", F.lit("202605"))
df_base = df_base.withColumn("LOTE", F.lit("BASE 2026-05-01"))
df_base = df_base.withColumn("cl_base", F.lit("Mayo 2026"))
df_base = df_base.withColumn("cl_carga", F.lit("2026-05-01"))
df_base = df_base.withColumn("cl_estado", F.lit("1"))


In [109]:
from pyspark.sql import functions as F

exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [110]:
df_score=df_score.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_base=df_base.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_base=df_base.withColumnRenamed('TIENDA','Tienda_IR')
df_base=df_base.withColumnRenamed('GRUPO_SEGMENTO_CREDICASH','GRUPO_SEGMENTO')
df_base=df_base.withColumnRenamed('REGION','Region')
df_base=df_base.withColumnRenamed('SEMAFORO','Semaforo')
df_base=df_base.withColumnRenamed('TIPDOC','TIPO_DOCUMENTO')
# df_base=df_base.withColumnRenamed('RANGO_SUELDO','CME_CREDICASH')
df_base=df_base.withColumnRenamed('TIPO_CLIENTE','SITUACION_LABORAL')
df_base=df_base.withColumnRenamed('RANGO_EDAD','DEMANDA')
df_base_inventario=df_base_inventario.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_base_inventario=df_base_inventario.withColumnRenamed('TIENDA','Tienda_IR')
df_base_inventario=df_base_inventario.withColumnRenamed('GRUPO_SEGMENTO_CREDICASH','GRUPO_SEGMENTO')
df_base_inventario=df_base_inventario.withColumnRenamed('REGION','Region')
df_base_inventario=df_base_inventario.withColumnRenamed('SEMAFORO','Semaforo')
df_base_inventario=df_base_inventario.withColumnRenamed('TIPDOC','TIPO_DOCUMENTO')
# df_base_inventario=df_base_inventario.withColumnRenamed('RANGO_SUELDO','CME_CREDICASH')
df_base_inventario=df_base_inventario.withColumnRenamed('RANGO_EDAD','DEMANDA')
df_base_inventario=df_base_inventario.withColumnRenamed('TIPO_CLIENTE','SITUACION_LABORAL')

In [111]:
print(df_base.columns)
print(df_base_inventario.columns)
print(df_score.columns)

['TIPO_DOCUMENTO', 'NUMERO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'DEMANDA', 'RANGO_SUELDO', 'SITUACION_LABORAL', 'INTENSIDAD_MAX']
['TIPO_DOCUMENTO', 'NUMERO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCUR

In [78]:
df_score.select('NUMERO_DOCUMENTO').count()

41194

In [79]:
df_score.select('NUMERO_DOCUMENTO').dropDuplicates(['NUMERO_DOCUMENTO']).count()

41194

In [ ]:
df_base_inventarioCME_CREDICASH

['TIPO_DOCUMENTO', 'NUMERO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'DEMANDA', 'CME_CREDICASH', 'SITUACION_LABORAL', 'INTENSIDAD_MAX']


In [ ]:
['TIPO_DOCUMENTO', 'NUMERO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'DEMANDA', 'CME_CREDICASH', 'SITUACION_LABORAL', 'INTENSIDAD_MAX']CME_CREDICASH

In [112]:
df_score=df_score.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_score=df_score.withColumnRenamed('CET','cl_telf1')
df_score=df_score.withColumnRenamed('CEL1','cl_telf2')

df_score = df_score.withColumn(
    "NUMERO_DOCUMENTO",
    F.right(
        F.concat(F.lit("00000000"), F.col("NUMERO_DOCUMENTO")),
        F.lit(8)
    )
)
df_base=df_base.join(df_score,['NUMERO_DOCUMENTO'],'left')
df_base_inventario=df_base_inventario.join(df_score,['NUMERO_DOCUMENTO'],'left')


In [ ]:
# df_base_inventario=df_base_inventario.withColumnRenamed('RANGO_EDAD','DEMANDA')
# TIPO_CLIENTE  por el momento lo colocare en demanda

In [113]:
df_base = df_base.withColumn("Campana", F.lit("202605"))
df_base = df_base.withColumn("LOTE", F.lit("BASE 2026-05-01"))
df_base = df_base.withColumn("cl_base", F.lit("Mayo 2026"))
df_base = df_base.withColumn("cl_carga", F.lit("2026-05-01"))
df_base = df_base.withColumn("cl_estado", F.lit("1"))

df_base_inventario = df_base_inventario.withColumn("Campana", F.lit("202605"))
df_base_inventario = df_base_inventario.withColumn("LOTE", F.lit("INVENTARIO"))
df_base_inventario = df_base_inventario.withColumn("cl_base", F.lit("Mayo 2026"))
df_base_inventario = df_base_inventario.withColumn("cl_carga", F.lit("2026-05-01"))
df_base_inventario = df_base_inventario.withColumn("cl_estado", F.lit("1"))



['NUMERO_DOCUMENTO', 'TIPO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'DEMANDA', 'CME_CREDICASH', 'SITUACION_LABORAL', 'INTENSIDAD_MAX', 'SCORE_TELEFONO', 'cl_telf1', 'cl_telf2']


In [ ]:
# df_01 = df_base.select(
#     "NUMERO_DOCUMENTO",
#     F.lit(1).alias("n")
# )

# df_02 = df_base_inventario.select(
#     "NUMERO_DOCUMENTO",
#     F.lit(2).alias("n")
# )

# df_0 = df_01.unionByName(df_02)

# window_spec = Window.partitionBy("NUMERO_DOCUMENTO").orderBy(F.col("n").asc())

# df_0 = (
#     df_0
#     .withColumn("rn", F.row_number().over(window_spec))
#     .filter(F.col("rn")>1)
#     .drop("rn")
# )

# df_0.groupBy("n").count().orderBy("n").show()

+---+-----+
|  n|count|
+---+-----+
+---+-----+



In [114]:
df_formato_base=df_base.toPandas()
df_formato_base_inventario=df_base_inventario.toPandas()

In [115]:
print(df_formato_base.shape)
print(df_formato_base_inventario.shape)

(73716, 42)
(93326, 42)


In [117]:

import pandas as pd

df_formato_base['cl_telf1'] = df_formato_base['cl_telf1'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

df_formato_base['cl_telf2'] = df_formato_base['cl_telf2'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

df_formato_base['cl_telf2'] = df_formato_base['cl_telf2'].fillna(0)
df_formato_base['cl_telf1'] = df_formato_base['cl_telf1'].fillna(0)

In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'fc_credicash_inventario.xlsx')
# df_formato_base_inventario.to_excel(ruta_archivo, index=False)

# ruta_archivo = os.path.join(ruta_csv, 'fc_credicash.xlsx')
# df_formato_base.to_excel(ruta_archivo, index=False)

In [147]:

ruta_archivo = os.path.join(ruta_csv, 'fc_alfin.csv')

df_formato_base.to_csv(
    ruta_archivo,
    index=False,
    sep=';'
)

subir

In [20]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT * FROM alfin_clientes
WHERE cl_base = 'abril 2026'
limit 10
"""

df_formato = pd.read_sql(query, engine_mysql)
print(df_formato.columns.tolist())

['cl_id', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'PILOTO_PLAZAS', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', '

In [175]:
filename='fc_alfin.csv'

filePath = os.path.join(ruta_csv, filename)

df_carga = pd.read_csv(filePath,sep=';')

df_carga["NUMERO_DOCUMENTO"] = (
    df_carga["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
print(df_carga.columns.tolist())


['DEPARTAMENTO', 'FLG_AAHH', 'NOMBRES', 'Agencia_comercial', 'CAPACIDAD_MAX', 'SALDO_DIFERENCIAL_REENG', 'GRUPO_MONTO', 'RANGO_OFERTA', 'cl_carga', 'NUMERO_DOCUMENTO', 'PLAZO', 'APELLIDO_PATERNO', 'SCORE_TELEFONO', 'RANGO_EDAD', 'oferta_max', 'cl_base', 'TIPO_CLIENTE', 'cl_estado', 'TIENDA', 'color_final', 'GRUPO_TASA', 'lote', 'Tasa_6', 'Tasa_5', 'cl_telf2', 'cl_telf1', 'Tasa_3', 'Tasa_7', 'flag_deuda_v_oferta', 'campania', 'USER_V3', 'TIPO_BASE', 'PROVINCIA', 'RANGO_SUELDO', 'DISTRITO', 'Region_comercial', 'FRESCURA', 'tipo_cliente_riegos', 'PROPENSION', 'nombre_base', 'Tasa_2', 'Tasa_4', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'MGNEG', 'Campana', 'CUOTA', 'estado', 'Edad', 'PROPENSION_IC', 'APELLIDO_MATERNO', 'Tasa_1']


In [89]:

df_formato_base["NUMERO_DOCUMENTO"] = (
    df_formato_base["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)


In [90]:

df_formato_base_inventario["NUMERO_DOCUMENTO"] = (
    df_formato_base_inventario["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)


In [126]:
df_formato_base['FLG_PEARS'] = (
    df_formato_base['FLG_PEARS']
    .astype(str)
    .str.upper()
    .map({'FALSE': 0, 'TRUE': 1})
    .fillna(0)
    .astype(int)
)

df_formato_base_inventario['FLG_PEARS'] = (
    df_formato_base_inventario['FLG_PEARS']
    .astype(str)
    .str.upper()
    .map({'FALSE': 0, 'TRUE': 1})
    .fillna(0)
    .astype(int)
)

In [125]:
df_formato_base['FLG_PEARS'] = df_formato_base['FLG_PEARS'].astype(int)
df_formato_base_inventario['FLG_PEARS'] = df_formato_base_inventario['FLG_PEARS'].astype(int)

ValueError: invalid literal for int() with base 10: 'False'

In [128]:
df_formato_base = df_formato_base.drop(columns=['DEMANDA'])
df_formato_base_inventario = df_formato_base_inventario.drop(columns=['DEMANDA'])

In [118]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [130]:

df_formato_base.to_sql(
    name="alfcc_clientes",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

73716

In [122]:
print(df_formato_base_inventario.columns.tolist())

['NUMERO_DOCUMENTO', 'TIPO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'DEMANDA', 'SITUACION_LABORAL', 'INTENSIDAD_MAX', 'SCORE_TELEFONO', 'cl_telf1', 'cl_telf2', 'Campana', 'LOTE', 'cl_base', 'cl_carga', 'cl_estado']


In [124]:
df_formato_base[['FLG_PEARS']].head()

,FLG_PEARS
0,False
1,False
2,True
3,True
4,False


In [ ]:
['NUMERO_DOCUMENTO', 'TIPO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'FLG_PEARS', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'DEMANDA', 'SITUACION_LABORAL', 'INTENSIDAD_MAX', 'SCORE_TELEFONO', 'cl_telf1', 'cl_telf2', 'Campana', 'LOTE', 'cl_base', 'cl_carga', 'cl_estado']




In [ ]:
df_carga['len_campana'] = df_carga['Campana'].astype(str).str.len()

print(df_carga['len_campana'].max())

In [116]:
print(df_carga.columns.tolist())


['DEPARTAMENTO', 'FLG_AAHH', 'NOMBRES', 'Agencia_comercial', 'CAPACIDAD_MAX', 'SALDO_DIFERENCIAL_REENG', 'GRUPO_MONTO', 'RANGO_OFERTA', 'cl_carga', 'NUMERO_DOCUMENTO', 'PLAZO', 'APELLIDO_PATERNO', 'SCORE_TELEFONO', 'RANGO_EDAD', 'oferta_max', 'cl_base', 'TIPO_CLIENTE', 'cl_estado', 'TIENDA', 'color_final', 'GRUPO_TASA', 'lote', 'Tasa_6', 'Tasa_5', 'cl_telf2', 'cl_telf1', 'Tasa_3', 'Tasa_7', 'flag_deuda_v_oferta', 'TIPO_BASE', 'USER_V3', 'PROVINCIA', 'RANGO_SUELDO', 'DISTRITO', 'Region_comercial', 'FRESCURA', 'tipo_cliente_riegos', 'Tasa_2', 'Tasa_4', 'nombre_base', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'MGNEG', 'Campana', 'CUOTA', 'estado', 'Edad', 'PROPENSION_IC', 'APELLIDO_MATERNO', 'Tasa_1']


In [ ]:
# soverwrite_table_SQL(spark,df_prueba,f'ventas_dinners_rtiro_borrar____1',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [ ]:
df_score = df_score.withColumn(
    "NUMERO_DOCUMENTO",
    F.right(
        F.concat(F.lit("00000000"), F.col("NUMERO_DOCUMENTO")),
        F.lit(8)
    )
)

df_prueba = df_prueba.withColumn(
    "NUMERO_DOCUMENTO",
    F.right(
        F.concat(F.lit("00000000"), F.col("NUMERO_DOCUMENTO")),
        F.lit(8)
    )
)

df_prueba=df_prueba.drop('SCORE_TELEFONO')
df_score=df_score.select('NUMERO_DOCUMENTO', 'cl_telf1', 'cl_telf2', 'SCORE_TELEFONO')
print(df_prueba.columns)



['NUMERO_DOCUMENTO', 'PROPENSION_IC', 'USER_V3', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'NOMBRES', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'campania', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÃ‘A_ANTERIOR', 'VARIACION_TASA_CAMPAÃ‘A_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'TIENDA', 'SUCURSAL', 'LOTE', 'cl_base', 'cl_carga', 'ESTADO', 'CAMPANA', 'nombre_base', 'cl_telf1', 'cl_telf2', 'SCORE_TELEFONO']


In [ ]:
query = f"""
    select top(1)* 
    from openquery([192.168.3.90],'Select 
    ,0 as cl_telf1
    ,0 as cl_telf2
    ,0 as cl_telf3
    ,0 as cl_telf4
    ,0 as cl_movil
    ,0 as cl_telefono
    ,0 as cl_turno
    ,0 as cl_gestor
    ,0 as cl_asesor
    ,0 as cl_gestion
    ,0 as cl_estado
    ,0 as cl_hits
    ,0 as cl_prioridad
    ,0 as cl_orden
    ,0 as cl_predictivo
    ,0 as cl_tiempo
    ,0 as cl_area
    from alfin_clientes where cl_base="Abril 2026"');
    """
df_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [8]:
from pyspark.sql import functions as F

cols_patron = [
    'NUMERO_DOCUMENTO', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'NOMBRES',
    'SUCURSAL', 'TIENDA', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion',
    'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7',
    'PLAZO', 'campania', 'TEM', 'Desgravamen', 'CUOTA', 'Edad',
    'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M',
    'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M',
    'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M',
    'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M',
    'Validador_Telefono', 'Prioridad', 'Nombre_prioridad',
    'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3',
    'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'sucursal_comercial',
    'Agencia_comercial', 'Region_comercial', 'RANGO_EDAD', 'RANGO_OFERTA',
    'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'TIPO_GEST', 'TIPO_CLIENTE',
    'CLIENTE_NUEVO', 'GRUPO_TASA', 'GRUPO_MONTO', 'TASA_VS_MONTO',
    'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG',
    'color', 'color_final', 'PROPENSION', 'SEGMENTO_USER', 'USUARIO',
    'tipo_cliente_riegos', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS',
    'FRESCURA', 'LEAD_CALIDAD', 'USER_V3', 'TIPO_BASE',
    'flag_deuda_v_oferta', 'MGNEG', 'PROPENSION_IC', 'PERFIL_RO',
    'FLG_AAHH', 'SCORE_TELEFONO', 'NumEntidades', 'INTENSIDAD_MAX',
    'nombre_base', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4',
    'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9',
    'cl_telf10', 'LOTE', 'ESTADO', 'CAMPANA', 'cl_base', 'tasa_minima',
    'cl_carga', 'MES_DURACION_BASE', 'ANIO_DURACION_BASE', 'PERFIL_GLOBAL'
]

# Crear columnas faltantes
for col in cols_patron:
    if col not in df_prueba.columns:
        df_prueba = df_prueba.withColumn(col, F.lit(None))

# Seleccionar exactamente en ese orden
df_prueba = df_prueba.select(cols_patron)

In [37]:
df_prueba.show()

+----------------+----------------+-------------------+-------------------+--------------------+--------------------+----------+-----------+-----------------+------+------+------+------+------+------+------+-----+--------+----+-----------+-------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------+-------------+--------------------+------------------+--------------------+----------------+----------+------------+------------+-------------+----+---------+-------------+-------------+-------------+---------------+-------------+---------------------+-----------------------+----------+-----+---------------+----------+-------------+-------+-------------------+------------------------+--------------+--------+------------+--------------------+------

In [9]:



df_dni = df_prueba.toPandas()
import os

ruta_archivo = os.path.join(ruta_csv, 'BASE_ALFIN_20260418.xlsx')

df_dni.to_excel(ruta_archivo, index=False)

In [20]:
df_prueba.show(5)


+----------------+-------------+--------+----------------+----------------+-----------------+----------+-----------+-----+------+-------------+---------+----------------------+--------+-----------------------+-------------+---------------+---------+----------------+------------+-------------+----------+------------------+-----------------+--------------------+---------------------------------+-------------------------------+-------------------------+----------+-------------------+-------------+---------------+-----+------+------+------+------+------+------+------+----+----------+------------+------------+------+--------+------------+--------------+-------------+------------+--------+-------------+-------------+----------+---------+----------+------+-------+-----------+---------+---------+--------------+
|NUMERO_DOCUMENTO|PROPENSION_IC| USER_V3|APELLIDO_PATERNO|APELLIDO_MATERNO|          NOMBRES|OFERTA_MAX|tasa_minima|PLAZO| CUOTA|CAPACIDAD_MAX|TIPO_GEST|TIPO_CLIENTE_COMERCIAL|campania|

In [ ]:
query = f"""
select *
    FROM valentina.dbo.alfin_clientes
    where cl_base='abril 2026'
    """
df_base_p=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_base_p = df_base_p.toDF(*[c.lower() for c in df_base_p.columns])

In [13]:
df_base_p=df_base_p.select('numero_documento','cl_id','nombre')

In [ ]:
# query = f"""
# select *
#     FROM valentina.dbo.alfin_clientes
#     where cl_base='abril 2026'
#     """
# df_base_p=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
# df_base_p = df_base_p.toDF(*[c.lower() for c in df_base_p.columns])
# print(df_base_p.columns)


['cl_id', 'tipo_doi', 'numero_documento', 'nombres', 'apellido_paterno', 'apellido_materno', 'sucursal', 'tienda', 'departamento', 'provincia', 'distrito', 'fec_nacimiento', 'oferta_max', 'oferta_reen', 'tipo_verificacion', 'grupo_riesgo', 'proveedor', 'lote', 'estado', 'tasa_1', 'tasa_2', 'tasa_3', 'tasa_4', 'tasa_5', 'tasa_6', 'tasa_7', 'segmento', 'campana', 'plazo', 'tem', 'propension_ic', 'desgravamen', 'cuota', 'edad', 'oferta_12m', 'tasa_12m', 'desgravamen_12m', 'cuota_12m', 'oferta_18m', 'tasa_18m', 'desgravamen_18m', 'cuota_18m', 'oferta_24m', 'tasa_24m', 'desgravamen_24m', 'cuota_24m', 'oferta_36m', 'tasa_36m', 'desgravamen_36m', 'cuota_36m', 'validador_telefono', 'prioridad', 'nombre_prioridad', 'deuda_1', 'entidad_1', 'deuda_2', 'entidad_2', 'deuda_3', 'entidad_3', 'sucursal_comercial', 'agencia_comercial', 'region_comercial', 'ubicacion', 'ofertamaximasinseguro', 'color', 'color_final', 'propension', 'oferta_final', 'garantia', 'oferta_minima_paperless', 'rango_oferta', 'r

In [ ]:
['cl_id', 'tipo_doi', 'numero_documento', 'nombres', 'apellido_paterno', 'apellido_materno', 'sucursal', 'tienda', 'departamento', 'provincia', 'distrito', 'fec_nacimiento', 'oferta_max', 'oferta_reen', 'tipo_verificacion', 'grupo_riesgo', 'proveedor', 'lote', 'estado', 'tasa_1', 'tasa_2', 'tasa_3', 'tasa_4', 'tasa_5', 'tasa_6', 'tasa_7', 'segmento', 'campana', 'plazo', 'tem', 'propension_ic', 'desgravamen', 'cuota', 'edad', 'oferta_12m', 'tasa_12m', 'desgravamen_12m', 'cuota_12m', 'oferta_18m', 'tasa_18m', 'desgravamen_18m', 'cuota_18m', 'oferta_24m', 'tasa_24m', 'desgravamen_24m', 'cuota_24m', 'oferta_36m', 'tasa_36m', 'desgravamen_36m', 'cuota_36m', 'validador_telefono', 'prioridad', 'nombre_prioridad', 'deuda_1', 'entidad_1', 'deuda_2', 'entidad_2', 'deuda_3', 'entidad_3', 'sucursal_comercial', 'agencia_comercial', 'region_comercial', 'ubicacion', 'ofertamaximasinseguro', 'color', 'color_final', 'propension', 'oferta_final', 'garantia', 'oferta_minima_paperless', 'rango_oferta', 'rango_sueldo', 'capacidad_max', 'peer', 'prop_comer', 'tipo_gest', 'cliente_nuevo', 'grupo_tasa', 'nuevos_3m', 'nuevos_6m', 'nuevos_9m', 'nuevos_12m', 'nuevos_4m', 'grupo_monto', 'tasa_vs_monto', 'usuario', 'incremento_monto_riesgos', 'flg_deuda_plus', 'tipo_cliente_riegos', 'user_v3', 'lead_calidad', 'segmento_user', 'rango_edad', 'rango_oferta2', 'periodo', 'retiro_gest', 'mejor_tipificacion', 'status', 'fecha_sol', 'base', 'resultado', 'num_enriquecido', 'tipo_contacto', 'q_ventas', 'localidad', 'desembolsado', 'monto_desembolsado', 'sbi', 'cruce', 'prest_previo', 'id_cliente', 'rango_edad2', 'fecha_envio', 'tipo_bd', 'cod_bd', 'nomb_bd', 'mes_gestion', 'tipo_cliente', 'grupo_tasa_reenganche', 'saldo_diferencial_reeng', 'flag_reeng', 'retiro_desembolso', 'frescura', 'flag_deuda_v_oferta', 'mgneg', 'perfil_ro', 'tipo_base', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'cl_estado', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'promocion', 'promocion2', 'nombre_base', 'numentidades', 'p_banco', 'perfil_global', 'flg_aahh', 'score_telefono', 'piloto_plazas', 'intensidad_max']tasa


In [ ]:
df_base_p=df_base_p.select('numero_documento','cl_id', 'nombres', 'apellido_paterno')


In [16]:
from pyspark.sql.functions import lit

df1=df_prueba.select(
    'numero_documento',
    'TELEFONO',
    lit(12).alias('uno')
).join(
    df_base_p.select('cl_id','numero_documento','nombres', 'apellido_paterno'),
    ['numero_documento'],
    'inner'
)

{"ts": "2026-04-16 09:23:46.273", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `TELEFONO` cannot be resolved. Did you mean one of the following? [`TASA_NUEVA`, `Dni_cliente`, `numero_documento`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor35.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o60.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `TELEFONO` cannot be resolved. Did you mean one of the following? [`TASA_NUEVA`, `Dni_cliente`, `numero_documento`]. SQLSTATE: 42703;\n'Project [numero_documento#39, 'TELEFONO, 12 AS uno#1445]\n+- Project [Dni_cliente#36, TASA_NUEVA#37, right(concat(00000000, Dni_cliente#36), 

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `TELEFONO` cannot be resolved. Did you mean one of the following? [`TASA_NUEVA`, `Dni_cliente`, `numero_documento`]. SQLSTATE: 42703;
'Project [numero_documento#39, 'TELEFONO, 12 AS uno#1445]
+- Project [Dni_cliente#36, TASA_NUEVA#37, right(concat(00000000, Dni_cliente#36), 8) AS numero_documento#39]
   +- Relation [Dni_cliente#36,TASA_NUEVA#37] csv


In [ ]:
filename='ultimo_base_crfc_alfinedi.csv'

df_prueba=cargar_archivo_csv(spark,filename,';',True)
# print(df_base_p.columns)
print(df_prueba.columns)

['cel', 'DNI']


In [24]:
df_prueba = df_prueba.withColumn(
    "numero_documento",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
)


In [ ]:
# overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_rtiro_borrar____1',server_sa,user_sa,pwd_sa,'ODIN')


In [25]:
print(df_base_p.columns)
print(df_prueba.columns)

['numero_documento', 'cl_id', 'nombre']
['cel', 'DNI', 'numero_documento']


In [26]:
from pyspark.sql.functions import lit

df1=df_prueba.select(
    'numero_documento',
    'cel',
    lit(17).alias('uno')
).join(
    df_base_p.select('cl_id','numero_documento','nombre'),
    ['numero_documento'],
    'inner'
)

In [27]:
print(df1.columns)


['numero_documento', 'cel', 'uno', 'cl_id', 'nombre']


In [28]:
df1=df1.select('cel', 'cl_id', 'uno', 'nombre')
df1 = df1.withColumn(
    "col_final",
    F.concat_ws(",", "cel", "cl_id", "uno", "nombre")
)

df1 = df1.withColumn(
    "col_final",
    F.concat_ws("", "col_final", lit(","))
)
df1=df1.select('col_final')


In [29]:
import os
df_carga=df1.toPandas()

ruta_archivo = os.path.join(ruta_csv, 'va_01_base2.csv')
df_carga.to_csv(ruta_archivo, index=False, sep=",",header=False)
# df_dni.to_excel(ruta_archivo, index=False)

In [7]:
filename='fugas_dni.csv'

df_prueba=cargar_archivo_csv(spark,filename,';',True)
print(df_prueba.columns)

['dni']


In [ ]:
['ID PROVEEDOR', 'NOMBRES', 'APELLIDO PATERNO', 'APELLIDO MATERNO', 'TIPO DE DOCUMENTO', 'NUMERO_DOCUMENTO', 'TASA(TEA)', 'TEA_DF', 'TELEFONO_CASA1', 'TELEFONO_CASA2', 'TELEFONO_CASA3', 'TELEFONO_CASA4', 'TELEFONO_CASA5', 'TELEFONO_CASA6', 'TELEFONO_CASA7', 'TELEFONO_CASA8', 'TELEFONO_CASA9', 'TELEFONO_CASA10', 'NUEVO_GRUPO03', 'PROB_CONTACTO', 'DINERS_NRO1', 'DINERS_NRO2', 'DINERS_NRO3', 'DINERS_NRO4', 'DINERS_NRO5', 'OSIPTEL_NRO1_ENCRIP', 'OSIPTEL_NRO2_ENCRIP', 'OSIPTEL_NRO3_ENCRIP', 'OSIPTEL_NRO4_ENCRIP', 'OSIPTEL_NRO5_ENCRIP', 'DCP01', 'DCP02', 'DCP03', 'DCP04', 'DCP05', 'CELULAR01', 'CELULAR02', 'CELULAR03', 'CELULAR04', 'CELULAR05', 'CELULAR06', 'CELULAR07', 'CELULAR08', 'CELULAR09', 'CELULAR10', 'RECENCIA', 'PERFIL', 'PRODUCTO', 'LINEA_DIN', 'ID RESULTADO GESTION', 'TCEA_NEW', 'TCEA_CUOTAS_DF', 'LIMACALLAO', 'FECHA_NACIMIENTO', 'PROVINCIA', 'L_BCP', 'L_BBVA', 'L_IBK', 'L_SCO', 'L_BIF', 'L_CITI', 'L_FIN', 'L_RIP', 'L_CMR', 'L_CRESCO', 'L_CEN', 'L_AZT', 'L_UNO', 'L_GNB', 'L_EFE', 'L_COM', 'L_NAC', 'EX_SOCIO', 'LINEA_ANT_EX_SOCIO', 'ULT_TASA_EX_SOCIO', 'PRIORIDAD', 'BENCH', 'FUENTE', 'SEGMENTACION']

In [4]:
df_prueba=df_prueba.select('NUMERO_DOCUMENTO','SEGMENTACION')

In [8]:
df_prueba = df_prueba.withColumn(
    "numero_documento",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni")),
        F.lit(8)
    )
)

In [23]:
df_prueba=df_prueba.select('numero_documento')

In [9]:
overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_rtiro_borrar____1',server_sa,user_sa,pwd_sa,'ODIN')
overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_rtiro_borrar____1',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_rtiro_borrar____1',server_zeus,user_zeus,pwd_zeus,'ODIN')


In [ ]:
overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_segmentacion',server_sa,user_sa,pwd_sa,'ODIN')
overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_segmentacion',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba,f'ventas_dinners_segmentacion',server_zeus,user_zeus,pwd_zeus,'ODIN')


In [ ]:
fecha_mes_base='2026-04-01'
query = f"""
    SELECT 
    a.NUMERO_DOCUMENTO as dni_cliente,
    t.contacto,
    t.tipo_telf,
    a.cl_carga
    FROM VALENTINA.dbo.alfin_clientes a
    CROSS APPLY (
    VALUES
        (a.cl_telf1, 'cel01'),
        (a.cl_telf2, 'cel02'),
        (a.cl_telf3, 'cel03'),
        (a.cl_telf4, 'cel04'),
        (a.cl_telf5, 'cel05'),
        (a.cl_telf6, 'cel06'),
        (a.cl_telf7, 'cel07'),
        (a.cl_telf8, 'cel08'),
        (a.cl_telf9, 'cel09'),
        (a.cl_telf10, 'cel10')
    ) t(contacto, tipo_telf)
    WHERE 
    t.contacto IS NOT NULL
    AND t.contacto <> ''
    AND LEN(t.contacto) = 9
    AND t.contacto LIKE '9%'
    AND CAST(a.cl_carga AS DATE) >= CAST('{fecha_mes_base}' AS DATE)
    AND CAST(a.cl_carga AS DATE) < DATEADD(MONTH, 1, CAST('{fecha_mes_base}' AS DATE))
    """

df_tnumero=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)


In [43]:

query = """
  SELECT * FROM cronox.dbo.borrar_df_cencosud_tc_01
    """
df_list_cencosud = obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

# df_list_cencosud = df_list_cencosud.withColumn(
#     'linea_sae',
#     col('linea_sae').cast('int')
# )
# df_list_cencosud = df_list_cencosud.withColumn(
#     'q_intentos_telef',
#     col('q_intentos_telef').cast('int')
# )
# df_list_cencosud = df_list_cencosud.withColumn(
#     'tea',
#     col('tea').cast('double')
# )


print(df_list_cencosud.columns)
print(df_list_cencosud.count())

['vendor_lead_code', 'phone_number', 'phone_number_02', 'first_name', 'last_name', 'address1', 'address2', 'address3', 'city', 'province', 'email', 'security_phrase', 'comments', 'mejor_CODIGO_cli_dia', 'mejor15_CODIGO_telf', 'mejor15_CODIGO_cli', 'mejor_CODIGO_cli', 'indice_num', 'tipo_telf', 'cod_attempt', 'q_intentos_telef', 'indice_tpo_telf', 'min_numero', 'phone_number_04', 'indice', 'segmento', 'tipdoc', 'regimen_laboral', 'quintil', 'linea_cencosud', 'rng_edad', 'propension_efectivo', 'tipo_tarjeta_cenco', 'num_tc', 'mejora_oferta', 'seg_contact', 'fecha_envio', 'antiguedad', 'retiro', 'seg_edad', 'flg_ex_cliente', 'region', 'flg_frescura', 'flg_cliente_nuevo', 'mes_prioridad', 'retiro_01', 'numero_campana', 'nombre_campana', 'fecha_hora_llamada', 'duracion', 'call_result', 'list_name', 'fecha_agenda', 'comentarios', 'fecha_llamada', 'tramo', 'dni_ejecutivo', 'ejecutivo', 'ult_call_result', 'dni_unico', 'mejor_estado_tipi_cli', 'mejor_descripcion_cli', 'mejor_sub_descripcion', '

In [13]:
df_list_cencosud.dropDuplicates(['vendor_lead_code']) \
    .groupBy("quintil") \
    .count() \
    .orderBy("quintil") \
    .show()

+-------+-----+
|quintil|count|
+-------+-----+
|      1| 7706|
|      2| 7305|
|      3| 7784|
|      4| 7560|
|      5| 7927|
+-------+-----+



In [ ]:
['vendor_lead_code', 'phone_number', 'phone_number_02', 'first_name', 'last_name', 'address1', 'address2', 'address3', 'city', 'province', 'email', 'security_phrase', 'comments', 'mejor_CODIGO_cli_dia', 'mejor15_CODIGO_telf', 'mejor15_CODIGO_cli', 'mejor_CODIGO_cli', 'indice_num', 'tipo_telf', 'cod_attempt', 'q_intentos_telef', 'indice_tpo_telf', 'min_numero', 'phone_number_04', 'indice', 'segmento', 'tipdoc', 'regimen_laboral', 'quintil', 'linea_cencosud', 'rng_edad', 'propension_efectivo', 'tipo_tarjeta_cenco', 'num_tc', 'mejora_oferta', 'seg_contact', 'fecha_envio', 'antiguedad', 'retiro', 'seg_edad', 'flg_ex_cliente', 'region', 'flg_frescura', 'flg_cliente_nuevo', 'mes_prioridad', 'retiro_01', 'numero_campana', 'nombre_campana', 'fecha_hora_llamada', 'duracion', 'call_result', 'list_name', 'fecha_agenda', 'comentarios', 'fecha_llamada', 'tramo', 'dni_ejecutivo', 'ejecutivo', 'ult_call_result', 'dni_unico', 'mejor_estado_tipi_cli', 'mejor_descripcion_cli', 'mejor_sub_descripcion', 'mejor_peso_cli', 'mejor15_estado_tipi_cli', 'mejor15_descripcion_cli', 'mejor15_sub_descripcion_cli', 'mejor15_peso_cli', 'mejor15_estado_tipi_telf', 'mejor15_descripcion_telf', 'mejor15_sub_descripcion_telf', 'mejor15_peso_telf', 'mejor_estado_tipi_cli_dia', 'mejor_descripcion_cli_dia', 'mejor_sub_descripcion_cli_dia', 'mejor_peso_cli_dia']esta


In [44]:
df_list_cencosud.dropDuplicates(['vendor_lead_code']).count()


38282

In [3]:
df_list_cencosud.dropDuplicates(['vendor_lead_code']).count()


38282

In [11]:
print([row['mejor15_descripcion_telf' ] for row in df_list_cencosud.select('mejor15_descripcion_telf').distinct().collect()])



['LLAMADA ELIMINADA POR ERROR EN RED (AUTO)', 'VOLVER A LLAMAR', 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA', 'NO UTILIZA (TARJETAS - PRESTAMOS)', 'AUTODIAL NO RESPONDE (AUTO)', 'OFERTA DE TASA MUY ALTA', 'OFERTA DE LINEA MUY BAJA', 'NO DESEA PAGAR MEMBRESIA', 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)', 'VOLVER A LLAMAR (TERCERO RELACIONADO)', 'ZONA FUERA DE COBERTURA', 'NO VOLVER A LLAMAR NUNCA MAS', 'VOLVER A LLAMAR - call', 'TELEFONO OCUPADO / NO CONTESTAN', 'OCUPADO (AUTO)', 'NO DESEA –NO ESPECIFICA MOTIVO', 'TELEFONO EQUIVOCADO', 'GESTION EN PROCESO (AUTO)', 'MENSAJE EN CASILLA DE VOZ (AUTO)', 'CLIENTE FALLECIO', 'CLIENTE ACEPTA PRODUCTO', 'TELEFONO FUERA DE SERVICIO / NO EXISTE', 'DESEA IR A AGENCIA', 'CLIENTE DESEA OTRO PRODUCTO', 'AGENTE NO DISPONIBLE (AUTO)', None]


In [6]:
flg_ex_cliente=['EX-CLIENTE', 'NUEVO']
tipo_telf=['BBDD CEL01']
segmento=['TC2', 'TC1', 'TC3']
first_name=['SEGMENTO SIN BONUS', 'SEGMENTO BONUS', 'SEGMENTO EXCLUSIVO']
regimen_laboral=['INDEP', 'DEP', 'INFOR']
tipo_tarjeta_cenco=['BLACK', 'PREMIUM', 'CLASICA']
quintil=[ '1', '2']
propension_efectivo=['1. TOMA EFECTIVO MAS TARJETA', '2. TOMA SOLO EFECTIVO', '3. TOMA SOLO TC']
mejor15_descripcion_telf=['LLAMADA ELIMINADA POR ERROR EN RED (AUTO)',
 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA',
 'NO UTILIZA (TARJETAS - PRESTAMOS)',
 'AUTODIAL NO RESPONDE (AUTO)',
 'OFERTA DE TASA MUY ALTA',
 'NO DESEA PAGAR MEMBRESIA',
 'OFERTA DE LINEA MUY BAJA',
 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)',
 'VOLVER A LLAMAR (TERCERO RELACIONADO)',
 'TELEFONO OCUPADO / NO CONTESTAN',
 'OCUPADO (AUTO)',
 'NO SE ASIGNO RESULTADO A LA LLAMADA (AUTO)',
 'NO DESEA –NO ESPECIFICA MOTIVO',
 'GESTION EN PROCESO (AUTO)',
 'NUMERO DESCONECTADO (AUTO)',
 'MENSAJE EN CASILLA DE VOZ (AUTO)',
 'DESEA IR A AGENCIA',
 'CLIENTE DESEA OTRO PRODUCTO',
 'AGENTE NO DISPONIBLE (AUTO)']

In [48]:
flg_ex_cliente=[ 'NUEVO']
tipo_telf=['bbdd cel01']
segmento=['TC2', 'TC1']
first_name=['SEGMENTO SIN BONUS', 'SEGMENTO BONUS', 'SEGMENTO EXCLUSIVO']
regimen_laboral=['', 'DEP', 'INFOR']
tipo_tarjeta_cenco=['BLACK', 'PREMIUM', 'CLASICA']
quintil=[ '1','2','3']
propension_efectivo=['1. TOMA EFECTIVO MAS TARJETA',  '3. TOMA SOLO TC']
mejor15_descripcion_telf=[
'LLAMADA ELIMINADA POR ERROR EN RED (AUTO)',
 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA',
 'NO UTILIZA (TARJETAS - PRESTAMOS)',
 'AUTODIAL NO RESPONDE (AUTO)',
 'OFERTA DE TASA MUY ALTA',
 'OFERTA DE LINEA MUY BAJA',
 'NO DESEA PAGAR MEMBRESIA',
 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)',
 'VOLVER A LLAMAR (TERCERO RELACIONADO)',
 'TELEFONO OCUPADO / NO CONTESTAN',
 'OCUPADO (AUTO)',
 'NO DESEA –NO ESPECIFICA MOTIVO',
 'GESTION EN PROCESO (AUTO)',
 'MENSAJE EN CASILLA DE VOZ (AUTO)',
 'DESEA IR A AGENCIA',
 'CLIENTE DESEA OTRO PRODUCTO',
 'AGENTE NO DISPONIBLE (AUTO)'
 ]

In [49]:
# mejor_tipi=['VOLVER A LLAMAR','VOLVER A LLAMAR - call']

df_filtrado = df_list_cencosud.filter(
    # ((F.col('mejor_estado_tipi_cli').isin('CONTACTO EFECTIVO','NO GESTIONADO')) | (F.col('mejor_estado_tipi_cli').isNull())) &
    ((F.col('mejor_descripcion_cli').isin(mejor15_descripcion_telf)) | (F.col('mejor_descripcion_cli').isNull())) &
    ((F.col('mejor15_descripcion_telf').isin(mejor15_descripcion_telf)) | (F.col('mejor15_descripcion_telf').isNull())) &
    ((F.col('fecha_llamada')<'2026-04-07') | (F.col('fecha_llamada').isNull())) &
    (F.col('quintil').isin(quintil))&
    (F.col('q_intentos_telef')<4)

    
).orderBy(col('linea_cencosud').desc())


print(df_filtrado.count())
print(df_filtrado.columns)

1015
['vendor_lead_code', 'phone_number', 'phone_number_02', 'first_name', 'last_name', 'address1', 'address2', 'address3', 'city', 'province', 'email', 'security_phrase', 'comments', 'mejor_CODIGO_cli_dia', 'mejor15_CODIGO_telf', 'mejor15_CODIGO_cli', 'mejor_CODIGO_cli', 'indice_num', 'tipo_telf', 'cod_attempt', 'q_intentos_telef', 'indice_tpo_telf', 'min_numero', 'phone_number_04', 'indice', 'segmento', 'tipdoc', 'regimen_laboral', 'quintil', 'linea_cencosud', 'rng_edad', 'propension_efectivo', 'tipo_tarjeta_cenco', 'num_tc', 'mejora_oferta', 'seg_contact', 'fecha_envio', 'antiguedad', 'retiro', 'seg_edad', 'flg_ex_cliente', 'region', 'flg_frescura', 'flg_cliente_nuevo', 'mes_prioridad', 'retiro_01', 'numero_campana', 'nombre_campana', 'fecha_hora_llamada', 'duracion', 'call_result', 'list_name', 'fecha_agenda', 'comentarios', 'fecha_llamada', 'tramo', 'dni_ejecutivo', 'ejecutivo', 'ult_call_result', 'dni_unico', 'mejor_estado_tipi_cli', 'mejor_descripcion_cli', 'mejor_sub_descripcio

In [50]:
tipificaicon_telf="mejor15_descripcion_telf"
df_filtrado = df_filtrado.select(
    'vendor_lead_code', 'phone_number', 'phone_number_02', 'first_name',
    'last_name', 'address1', 'address2', 'address3',
    'city', 'province', 'email', 'security_phrase', 'comments','mejor15_descripcion_telf'
)

In [51]:
df_dni = df_filtrado.toPandas()

import os

ruta_archivo = os.path.join(ruta_csv, 'tc_cenco_202604080941.xlsx')

df_dni.to_excel(ruta_archivo, index=False)